# Лабораторная работа 1. Морфологический анализ

### Задание 1.
1. Изучите документацию и лицензию морфологического парсера [pymorphy2](https://pymorphy2.readthedocs.io).
1. Установите `pymorphy2`.

In [ ]:
# Если библиотека уже установлена, эту ячейку можно пропустить.
%pip install -q pymorphy2

In [ ]:
import json
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pymorphy2

In [ ]:
MORPH = pymorphy2.MorphAnalyzer()

### Задание 2.
Напишите функцию `parse_text()`, на вход которых поступает текст (в виде строки), а на выходе формируется список, содержащий для каждого слова входного текста следующую информацию:
- исходную словоформу (`wordform`);
- нормальную форму слова (лемму) (`norm` или `lemma`);
- часть речи (part of speech, `pos`);
- другую грамматическую информацию, выдаваемую `pymorphy2`;
- признак, присутствует ли слово в словаре `pymorphy2`.

Функция должна выбирать наиболее вероятный вариант морфологического разбора слова.


In [ ]:
def parse_text(text: str):
    """Разбирает текст и возвращает список словарей с морфологической информацией."""
    tokens = re.findall(r"[A-Za-zА-Яа-яЁё-]+", text)
    parsed = []

    for token in tokens:
        best = MORPH.parse(token)[0]
        parsed.append(
            {
                "wordform": token,
                "lemma": best.normal_form,
                "pos": best.tag.POS,
                "tag": str(best.tag),
                "is_known": bool(best.is_known),
            }
        )

    return parsed

### Задание 3.
Напишите функцию `save_morph_results()`, сохраняющую структуру данных, получаемую функцией `parse_text()`, в текстовый файл формата JSON.

In [ ]:
def save_morph_results(results, out_path: str):
    """Сохраняет результаты parse_text() в JSON."""
    out_file = Path(out_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)

    with out_file.open("w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    return out_file

### Задание 4.
Напишите функцию `get_dictionary()`, на вход которой поступает структура данных, получаемая функцией `parse_text()`, а на выходе формируется словарь, ключами которого являются все нормальные формы слов текста, а в качестве значений хранится следующая информация:
- часть речи слова;
- частота слова в тексте;
- все варианты словоформ в тексте с данной нормальной формой.

In [ ]:
def get_dictionary(parsed_data):
    """Строит словарь по леммам: часть речи, частота, словоформы."""
    dictionary = {}

    by_lemma = defaultdict(list)
    for item in parsed_data:
        by_lemma[item["lemma"]].append(item)

    for lemma, items in by_lemma.items():
        forms_counter = Counter(i["wordform"] for i in items)
        pos_counter = Counter(i["pos"] for i in items)
        most_common_pos = pos_counter.most_common(1)[0][0]

        dictionary[lemma] = {
            "pos": most_common_pos,
            "freq": len(items),
            "wordforms": dict(forms_counter),
        }

    return dictionary

### Задание 5.
Напишите функцию `save_dictionary()`, сохраняющую предыдущую структуру данных в текстовый файл формата JSON.
Слова в файле должны быть упорядочены по убыванию частоты.

In [ ]:
def save_dictionary(dictionary_data, out_path: str):
    """Сохраняет словарь лемм в JSON с сортировкой по убыванию частоты."""
    out_file = Path(out_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)

    sorted_items = sorted(
        dictionary_data.items(),
        key=lambda x: x[1]["freq"],
        reverse=True,
    )
    sorted_dictionary = {lemma: info for lemma, info in sorted_items}

    with out_file.open("w", encoding="utf-8") as f:
        json.dump(sorted_dictionary, f, ensure_ascii=False, indent=2)

    return out_file

### Задание 6.
Напишите функцию `get_non_dict()`, на вход которой поступает структура данных, получаемая функцией `parse_text()`, а на выходе формируется словарь, содержащий слова текста, отсутствующие в словаре `pymorphy2`, вместе с частотой слова в тексте.

In [ ]:
def get_non_dict(parsed_data):
    """Возвращает словарь слов, отсутствующих в словаре pymorphy2."""
    counter = Counter()
    for item in parsed_data:
        if not item["is_known"]:
            counter[item["wordform"]] += 1

    return dict(counter.most_common())

### Задание 7.
Напишите функцию `save_non_dict()`, сохраняющую структуру данных, получаемую функцией `get_non_dict()`, в текстовый файл формата TSV (tab-separated values).
Слова в файле должны быть упорядочены по убыванию частоты.

In [ ]:
def save_non_dict(non_dict_data, out_path: str):
    """Сохраняет неизвестные слова в TSV, отсортированные по частоте."""
    out_file = Path(out_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)

    sorted_items = sorted(non_dict_data.items(), key=lambda x: x[1], reverse=True)

    with out_file.open("w", encoding="utf-8") as f:
        f.write("word\tfreq\n")
        for word, freq in sorted_items:
            f.write(f"{word}\t{freq}\n")

    return out_file

### Задание 8.
Напишите функцию `get_pos_distribution()`, на вход которой поступает словарь, формируемый функцией `get_dictionary()`, а на выходе выдается структура данных, содержащая частотное распределение частей речи в словаре:
- часть речи – количество уникальных слов – общее количество слов.

In [ ]:
def get_pos_distribution(dictionary_data):
    """Возвращает распределение частей речи: уникальные слова и токены."""
    distribution = defaultdict(lambda: {"unique_words": 0, "total_tokens": 0})

    for _, info in dictionary_data.items():
        pos = info["pos"] or "UNKN"
        distribution[pos]["unique_words"] += 1
        distribution[pos]["total_tokens"] += info["freq"]

    return dict(distribution)

### Задание 9.
Создайте текстовую коллекцию - скачайте не менее **10 текстов** разных жанров и разного размера (например, произведения классиков, современных писателей, новостные статьи, научные статьи и т.п.).

Учитывайте кодировку – все файлы должны быть в UTF-8.

In [ ]:
def read_text_collection(texts_dir="texts"):
    """Читает все .txt файлы из папки и возвращает {имя_файла: текст}."""
    texts = {}
    for path in sorted(Path(texts_dir).glob("*.txt")):
        texts[path.name] = path.read_text(encoding="utf-8")
    return texts

# Ваш код для открытия файлов.
collection = read_text_collection("texts")
len(collection), list(collection)[:3]

### Задание 10.
Обработайте текстовую коллекцию при помощи функций `parse_text()`, `get_dictionary()` и `get_non_dict()`, и сохраните результаты в текстовых файлах при помощи функций `save_morph_results()`, `save_dictionary()` и `save_non_dict()`.

Для каждого входного текстового файла выведите на экран следующую информацию:
- имя файла;
- размер файла в байтах;
- размер текста в словах (общее количество слов в тексте);
- размер словаря (количество уникальных слов в тексте);
- время обработки файла (в секундах).

In [ ]:
def process_collection(collection, output_dir="results"):
    """Обрабатывает коллекцию текстов и возвращает таблицу со статистикой."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    dictionaries = {}

    for file_name, text in collection.items():
        t0 = time.perf_counter()

        parsed = parse_text(text)
        dictionary = get_dictionary(parsed)
        non_dict = get_non_dict(parsed)

        stem = Path(file_name).stem
        save_morph_results(parsed, output_dir / f"{stem}_morph.json")
        save_dictionary(dictionary, output_dir / f"{stem}_dict.json")
        save_non_dict(non_dict, output_dir / f"{stem}_non_dict.tsv")

        elapsed = time.perf_counter() - t0

        rows.append(
            {
                "file": file_name,
                "size_bytes": len(text.encode("utf-8")),
                "tokens_count": len(parsed),
                "vocab_size": len(dictionary),
                "time_sec": elapsed,
            }
        )

        dictionaries[file_name] = dictionary

    report = pd.DataFrame(rows).sort_values("size_bytes")
    return report, dictionaries

# Ваш код для обработки коллекции и вывода результатов
report, dictionaries = process_collection(collection, output_dir="results")
report

### Задание 11.
Для самого большого словаря выполните следующее:
- постройте частотное распределение слов: по оси ординат – частота, по оси абсцисс – слова, упорядоченные по убыванию частоты (по-другому, *ранги* слов). В качестве значений на оси абсцисс можно выводить ранги слов, а не сами слова;
- постройте в виде диаграммы распределение частей речи, полученное функцией `get_pos_distribution()`.

In [ ]:
def plot_largest_dictionary(dictionaries):
    """Строит график рангового распределения слов и диаграмму частей речи."""
    if not dictionaries:
        raise ValueError("Словари отсутствуют: сначала обработайте коллекцию.")

    largest_file = max(dictionaries, key=lambda k: len(dictionaries[k]))
    largest_dict = dictionaries[largest_file]

    freqs = sorted((info["freq"] for info in largest_dict.values()), reverse=True)
    ranks = list(range(1, len(freqs) + 1))

    plt.figure(figsize=(10, 4))
    plt.plot(ranks, freqs)
    plt.title(f"Ранговое распределение слов: {largest_file}")
    plt.xlabel("Ранг")
    plt.ylabel("Частота")
    plt.grid(alpha=0.3)
    plt.show()

    pos_dist = get_pos_distribution(largest_dict)
    pos_labels = list(pos_dist.keys())
    pos_values = [v["total_tokens"] for v in pos_dist.values()]

    plt.figure(figsize=(8, 6))
    plt.pie(pos_values, labels=pos_labels, autopct="%.1f%%", startangle=90)
    plt.title(f"Распределение частей речи: {largest_file}")
    plt.show()

# Ваш код для вывода графика и диаграммы
plot_largest_dictionary(dictionaries)

### Задание 12.
Постройте график зависимость времени морфологического анализа от размера текстового файла.

In [ ]:
def plot_time_vs_size(report_df):
    """Строит график зависимости времени анализа от размера файла."""
    if report_df.empty:
        raise ValueError("Отчёт пуст: сначала обработайте коллекцию.")

    plt.figure(figsize=(8, 5))
    plt.scatter(report_df["size_bytes"], report_df["time_sec"])

    for _, row in report_df.iterrows():
        plt.annotate(row["file"], (row["size_bytes"], row["time_sec"]), fontsize=8)

    plt.title("Время морфологического анализа vs размер файла")
    plt.xlabel("Размер файла (байт)")
    plt.ylabel("Время (сек)")
    plt.grid(alpha=0.3)
    plt.show()

# Ваш код для вывода графика
plot_time_vs_size(report)